# Preprocessing

In [1]:
import pandas as pd

products = pd.read_csv("/kaggle/input/datasets/sadmanshafiq1611/products-new/2_product_details.csv")

print(products.head())
print(products.shape)
print(products.info())
print(products.describe())


                                    URL  \
0  https://www.amazon.com/dp/B0GJ5KNC27   
1  https://www.amazon.com/dp/B0FRN2Z4JF   
2  https://www.amazon.com/dp/B09C13PZX7   
3  https://www.amazon.com/dp/B0G12HGHGM   
4  https://www.amazon.com/dp/B0BX52C4LN   

                                               Title       Brand Rating  \
0  GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...   GravaStar    4.5   
1  GravaStar Mercury M1 Pro Wireless Gaming Mouse...   GravaStar    4.5   
2  Razer Basilisk V3 Customizable RGB Wired Ergon...       Razer    4.6   
3  Logitech G PRO X2 SUPERSTRIKE Wireless Gaming ...  Logitech G    4.5   
4  HyperX Pulsefire Haste 2 – Wireless Gaming Mou...      HyperX    4.1   

    Price Review_Count                                          Image_URL  \
0   71.95        2,070  https://m.media-amazon.com/images/I/61M5yEjuIF...   
1   89.96        1,485  https://m.media-amazon.com/images/I/51BhB+GhmQ...   
2   34.98       10,516  https://m.media-amazon.com/images/I

In [2]:
print(len(products))
products = products.drop_duplicates(subset="URL", keep="first")
print(len(products))
products = products[products["Price"] != "not available"]

print(len(products))

products = products[products["Rating"] != "not available"]

print(len(products))

products = products[products["Brand"] != "not available"]

print(len(products))

products = products[products["Category"] != "not available"]

print(len(products))

products = products[products["Description"] != "not available"]


print(len(products))

1162
1152
1136
1129
1129
1125
1043


In [3]:
# products["Category"] = products["Category"].str.split(",").str[-1]
# print(products["Category"].nunique())

In [4]:
products["Price"] = products["Price"].str.replace("..", ".", regex=False)
products["Price"] = pd.to_numeric(products["Price"],errors="coerce")

products = products.dropna()
print(len(products))

1040


In [5]:
# def tier(x):
#     return pd.qcut(x, 3, labels=["Low", "Mid", "High"])

# products["price_tier"] = products.groupby("Category")["Price"].transform(tier)


products["price_tier"] = pd.qcut(products["Price"],q=3,labels=["budget", "mid-range", "premium"])
print(products.head())

                                    URL  \
0  https://www.amazon.com/dp/B0GJ5KNC27   
1  https://www.amazon.com/dp/B0FRN2Z4JF   
2  https://www.amazon.com/dp/B09C13PZX7   
3  https://www.amazon.com/dp/B0G12HGHGM   
4  https://www.amazon.com/dp/B0BX52C4LN   

                                               Title       Brand Rating  \
0  GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...   GravaStar    4.5   
1  GravaStar Mercury M1 Pro Wireless Gaming Mouse...   GravaStar    4.5   
2  Razer Basilisk V3 Customizable RGB Wired Ergon...       Razer    4.6   
3  Logitech G PRO X2 SUPERSTRIKE Wireless Gaming ...  Logitech G    4.5   
4  HyperX Pulsefire Haste 2 – Wireless Gaming Mou...      HyperX    4.1   

    Price Review_Count                                          Image_URL  \
0   71.95        2,070  https://m.media-amazon.com/images/I/61M5yEjuIF...   
1   89.96        1,485  https://m.media-amazon.com/images/I/51BhB+GhmQ...   
2   34.98       10,516  https://m.media-amazon.com/images/I

# Feature Engineering

In [6]:
def convert_to_lower(text):
    text = str(text).lower()
    return text

combined_text = []

for index, row in products.iterrows():
    combined = row["Title"] + " " + row["Description"] + " " + row["Brand"]
    combined_text.append(convert_to_lower(combined))

products["Combined_Text"] = combined_text

In [7]:
products.columns

products = products.drop(columns=["URL", "Price", "Title"])
products.columns


Index(['Brand', 'Rating', 'Review_Count', 'Image_URL', 'Description',
       'Category', 'Reviews', 'asin', 'keyword', 'price_tier',
       'Combined_Text'],
      dtype='object')

In [8]:
products["Rating"] = pd.to_numeric(products["Rating"], errors="coerce")

products["Review_Count"] = products["Review_Count"].str.replace(",", "", regex=False)
products["Review_Count"] = pd.to_numeric(products["Review_Count"], errors="coerce")

products = products.dropna()
print(len(products))

1040


# Model Selection and Training

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = products[["Combined_Text", "Rating", "Review_Count"]]
y = products["price_tier"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(832, 3)
(208, 3)


In [10]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = vectorizer.fit_transform(X_train["Combined_Text"])
X_test_tfidf = vectorizer.transform(X_test["Combined_Text"])

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(832, 5000)
(208, 5000)


In [11]:
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

scaler = MinMaxScaler()

X_train_num = scaler.fit_transform(X_train[["Rating", "Review_Count"]])
X_test_num = scaler.transform(X_test[["Rating", "Review_Count"]])

X_train_combined = hstack([X_train_tfidf, X_train_num])
X_test_combined = hstack([X_test_tfidf, X_test_num])

print(X_train_combined.shape)
print(X_test_combined.shape)

(832, 5002)
(208, 5002)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_combined, y_train)

y_pred = log_reg.predict(X_test_combined)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6490384615384616
              precision    recall  f1-score   support

      budget       0.74      0.63      0.68        76
   mid-range       0.60      0.49      0.54        75
     premium       0.62      0.88      0.72        57

    accuracy                           0.65       208
   macro avg       0.65      0.67      0.65       208
weighted avg       0.65      0.65      0.64       208



In [13]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_combined, y_train)

y_pred = nb.predict(X_test_combined)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6346153846153846
              precision    recall  f1-score   support

      budget       0.74      0.66      0.69        76
   mid-range       0.61      0.44      0.51        75
     premium       0.57      0.86      0.69        57

    accuracy                           0.63       208
   macro avg       0.64      0.65      0.63       208
weighted avg       0.65      0.63      0.63       208



In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=65)
rf.fit(X_train_combined, y_train)

y_pred = rf.predict(X_test_combined)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6730769230769231
              precision    recall  f1-score   support

      budget       0.80      0.64      0.72        76
   mid-range       0.64      0.55      0.59        75
     premium       0.60      0.88      0.71        57

    accuracy                           0.67       208
   macro avg       0.68      0.69      0.67       208
weighted avg       0.69      0.67      0.67       208

